# Hello World: Remote Data Ingestion

**The 1-Minute LakeLogic Demo.**

[!["Open In Colab"](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/LineageLogic/LakeLogic/blob/main/examples/01_quickstart/01_hello_world.ipynb) 
[![GitHub Repo](https://img.shields.io/badge/GitHub-Repo-blue?logo=github)](https://github.com/LineageLogic/LakeLogic/blob/main/examples/01_quickstart/01_hello_world.ipynb)

## ⚡ Zero Setup, Pure Value
In this example, we'll demonstrate how LakeLogic can point at a **remote URL** and instantly deliver a governed Data Lakehouse table. 

No local databases, no file downloads, just pure extraction and validation.

In [ ]:
pip list

In [2]:
# %pip install lakelogic[all]
from lakelogic import DataProcessor
import os

# Use a public raw data URL
REMOTE_URL = "https://raw.githubusercontent.com/LineageLogic/LakeLogic/main/examples/03_data_sources/files/excel/data/employees.csv"

print(f"🌍 Target: {REMOTE_URL}")

🌍 Target: https://raw.githubusercontent.com/LineageLogic/LakeLogic/main/examples/03_data_sources/files/excel/data/employees.csv


## 🧪 1. Run via In-Memory Contract (Python Dict)
LakeLogic allows you to define contracts as **Python Dictionaries**. 

**Best For**: Prototyping, Dynamic rule generation, and "all-in-one" notebooks.

In [3]:
contract_dict = {
    "version": "1.0.0",
    "dataset": "remote_employees",
    "source": {"type": "landing"},
    "quality": {
        "row_rules": [
            {"name": "Valid Email", "sql": "email LIKE '%@%'"}
        ]
    }
}

# Initialize with the dictionary directly
processor = DataProcessor(contract=contract_dict)
result = processor.run_source(REMOTE_URL)

print(f"✅ Success! Raw records: {len(result.raw)}, Valid: {len(result.good)}")

2026-02-21 10:12:31.266 | INFO     | lakelogic.core.processor:run_source:531 - Loading source: https://raw.githubusercontent.com/LineageLogic/LakeLogic/main/examples/03_data_sources/files/excel/data/employees.csv via polars


OSError: object-store error: Object at location  not found: Error performing HEAD https://raw.githubusercontent.com/LineageLogic/LakeLogic/main/examples/03_data_sources/files/excel/data/employees.csv in 95.8814ms - Server returned non-2xx status code: 404 Not Found: 

## 📝 2. Run via External Contract (YAML File)
For production, we recommend storing contracts as **YAML files**. 

**Best For**: Version Control (Git), Shared team definitions, and Formal Governance.

In [3]:
import yaml

# 1. Save our contract to a file (simulating a git-tracked file)
with open("users_contract.yaml", "w") as f:
    yaml.dump(contract_dict, f)

# 2. Initialize LakeLogic by passing the PATH string
processor_prod = DataProcessor(contract="users_contract.yaml")
result_prod = processor_prod.run_source(REMOTE_URL)

print("✅ Success via YAML Path!")

2026-02-19 12:07:59.520 | INFO     | lakelogic.core.processor:run_source:410 - Loading source: https://raw.githubusercontent.com/LineageLogic/LakeLogic/main/examples/03_data_sources/files/excel/data/employees.csv via polars
2026-02-19 12:07:59.591 | INFO     | lakelogic.core.processor:run:280 - Starting LakeLogic run [Auto-Engine: polars, Contract: remote_employees]
2026-02-19 12:07:59.594 | INFO     | lakelogic.core.processor:run:321 - Run complete. Source: 5, Total (post-transform): 5, Good: 4, Quarantined: 1, Pre-Transform Dropped: 0, Ratio: 20.00%


✅ Success via YAML Path!


## 📊 3. Inspect Results
Regardless of how you load the contract, LakeLogic returns the same rich `ValidationResult` object.

In [4]:
print("🏆 CLEAN DATA (Validated & Ready for Analytics):")
display(result.good)

🏆 CLEAN DATA (Validated & Ready for Analytics):


id,name,email,department,salary,hire_date,status
i64,str,str,str,i64,str,str
1,"""Frank Wilson""","""frank@company.com""","""Engineering""",105000,"""2023-02-10""","""active"""
3,"""Henry Brown""","""henry@company.com""","""Marketing""",-10000,"""2023-06-20""","""inactive"""
4,"""Iris Taylor""","""iris@company.com""","""Sales""",92000,"""2022-12-05""","""active"""
5,"""Jack Anderson""","""jack@company.com""","""HR""",79000,"""2024-02-14""","""active"""


In [5]:
print("❌ BAD DATA (Invalidated & Ready for Quarantine/Review/Fix/Reprocessing):")
display(result.bad)

❌ BAD DATA (Invalidated & Ready for Quarantine/Review/Fix/Reprocessing):


id,name,email,department,salary,hire_date,status,_lakelogic_errors,_lakelogic_categories,quarantine_state,quarantine_reprocessed
i64,str,str,str,i64,str,str,list[str],list[str],str,bool
2,"""Grace Lee""","""grace-no-at""","""Finance""",88000,"""2023-04-15""","""active""","[""Rule failed: Valid Email (email LIKE '%@%')""]","[""correctness""]","""active""",false


## 🏁 Summary: Dictionary vs. YAML

| Approach | Best Use Case | Benefit |
| :--- | :--- | :--- |
| **Python Dict** | Prototyping / Notebooks | Fast iteration, no extra files |
| **YAML File** | Production / Enterprise | Git versioning, shared governance |